# **Практическое задание по дисциплине «Теория принятия решений»**
Вариант 56 (104)



# **Задача 1**

В цехах N1 и N2 предприятия производится продукт Y, который в дальнейшем используется
в качестве исходного материала для производства изделий в цехе N3. Суммарная производи
тельность цехов N1 и N2 зависит от вложения дополнительных средств X. При работе цехов N1
и N2 в течение одного месяца эта зависимость может быть приближенно представлена в виде
функций:

*   N1: y = 5 + (x+15)**(1/2)
*   N2: y = 2 + (x+10)**(1/3)

Функции остатка средств в течение месяца:

*   N1: 0.86x
*   N2: 0.91x

Средства,выделяемые на оба цеха в течение квартала (3 месяца), составляют 102 единиц;
перераспределение производится помесячно.

Требуется распределить средства на планируемый квартал с целью получения максималь
ного количества продукта Y.



# **Этапы, управления, состояния**

**Этап:**
n = 1,2,3 - номер месяца

**Управление:**
Un - средства, выделяемые цеху N1 в n месяце
1. Тогда цеху N2 выделяется Sn - Un
2. Ограничения: 0 <= Un <= Sn


**Состояние:**
Sn - сумма средств, имеющаяся в начале n месяца

Начальное состояние:
1. S1 = 102
2. Состояние неотрицательно

**Переход к следующему месяцу:**

Sn+1 = 0.86 * Un + 0.91 * (Sn - Un) = 0.91 * Sn - 0.05 * Un

**Выигрыш за n месяц:**

g(Un, Sn) = 5 + (Un+15)^(1/2) + 2 + ((Sn-Un)+10)^(1/3) = 7 + (Un+15)^(1/2) + (Sn - Un + 10)^(1/3)

**Граничное условие (последний месяц, n=3):**

1. W3(S) = max(0 <= u <= S)
2. (7 + (U+15)^(1/2) + (S-U+10)^(1/3))

**Рекуррентное соотношение для n=2,1:**

1. Wn(S) = max(0 <= u <= S)
2. (7 + (U+15)^(1/2) + (S-U+10)^(1/3) + W(n+1)(0.91 S - 0.05 u))


# **Надо найти W1(102) и оптимальные управления U1,U2,U3**

In [ ]:
import sys                                                                      # Устанавливаем numpy, чтобы не выводить сообщения
!{sys.executable} -m pip install numpy -q
import numpy as np



class ResourceAllocation:                                                       # Класс для решения задачи оптимального распределения средств между двумя цехами на три месяца
    def __init__(self, periods=3, start_capital=102, discretization_step=1.0):
        self.periods = periods                                                  # Периоды планирования в месяцах
        self.start_capital = start_capital                                      # Начальный капитал
        self.step = discretization_step                                         # Шаг дискретизации состояния и управления

        self.capital_levels = np.arange(0, start_capital + self.step, self.step)# Возможные значения капитала на начало месяца
        self.num_levels = len(self.capital_levels)                              # Количество возможных состояний
        self.bellman_table = None                                               # Таблица для хранения результатов значения функции Беллмана



    def monthly_profit(self, capital, investment_in_plant1):                    # Вычисляет суммарный выпуск продукта Y за месяц
        # Формула: 7 + (U+15)^(1/2) + (S - U + 10)^(1/3)
        return 7 + (investment_in_plant1 + 15) ** (1/2) + (capital - investment_in_plant1 + 10) ** (1/3)



    def next_capital(self, capital, investment_in_plant1):                      # Переход к следующему состоянию: остаток средств после месяца
        # Формула: 0.86*U + 0.91*(S-U) = 0.91*S - 0.05*U
        return 0.91 * capital - 0.05 * investment_in_plant1



    def feasible_investments(self, capital):                                    # Генератор допустимых значений вложений в N1
        u = 0.0
        while u <= capital + 1e-9:                                              # Небольшой допуск для корректного учета границы из за погрешностей
            yield u
            u += self.step



    def solve(self):                                                            # Основной метод, реализующий обратную прогонку динамического программирования
        self.bellman_table = [None] * self.periods                              # Инициализируем таблицу: для каждого периода и каждого состояния
        for period in range(self.periods):
            self.bellman_table[period] = [None] * self.num_levels

        last_period = self.periods - 1                                          # Последний период 3 месяца
        for idx, capital in enumerate(self.capital_levels):                     # Перебираем все возможные значения капитала на начало последнего месяца
            best_profit = -np.inf
            best_action = None
            for u in self.feasible_investments(capital):                        # Для каждого допустимого вложения в N1
                profit = self.monthly_profit(capital, u)
                if profit > best_profit:
                    best_profit = profit
                    best_action = u
            self.bellman_table[last_period][idx] = (best_profit, best_action)   # Сохраняем пару для данного состояния

        for period in range(last_period - 1, -1, -1):                           # Обратный ход для предыдущих периодов 1 и 2 месяца
            for idx, capital in enumerate(self.capital_levels):
                best_total = -np.inf
                best_action = None

                for u in self.feasible_investments(capital):                    # Перебираем возможные вложения в N1
                    next_cap = self.next_capital(capital, u)                    # Вычисляем состояние на следующий месяц
                    next_idx = int(round(next_cap / self.step))                 # Находим индекс ближайшего состояния в сетке
                    next_idx = max(0, min(next_idx, self.num_levels - 1))       # Ограничиваем индекс допустимыми границами
                    future_profit = self.bellman_table[period + 1][next_idx][0] # Будущий максимальный выпуск, начиная со следующего месяца
                    total = self.monthly_profit(capital, u) + future_profit     # Суммарный выпуск за текущий и оставшиеся месяцы
                    if total > best_total:
                        best_total = total
                        best_action = u

                self.bellman_table[period][idx] = (best_total, best_action)     # Сохраняем оптимальные значения для текущего периода и состояния

        start_idx = int(round(self.start_capital / self.step))                  # Начальное состояние
        max_total_profit = self.bellman_table[0][start_idx][0]
        print(f"Максимальный суммарный выпуск за 3 месяца: {max_total_profit:.4f}")

        capital = self.start_capital                                            # Восстанавливаем оптимальную траекторию
        print("Оптимальное распределение средств по месяцам:")
        for period in range(self.periods):
            idx = int(round(capital / self.step))
            action = self.bellman_table[period][idx][1]
            next_capital = self.next_capital(capital, action)
            print(f"Месяц {period+1}: в N1 = {action:.2f}, в N2 = {capital - action:.2f}, "
                  f"остаток на след. месяц = {next_capital:.2f}")
            capital = next_capital



# Запуск решения
solver = ResourceAllocation(periods=3, start_capital=102, discretization_step=1.0)
solver.solve()

Максимальный суммарный выпуск за 3 месяца: 58.2261
Оптимальное распределение средств по месяцам:
Месяц 1: в N1 = 86.00, в N2 = 16.00, остаток на след. месяц = 88.52
Месяц 2: в N1 = 82.00, в N2 = 6.52, остаток на след. месяц = 76.45
Месяц 3: в N1 = 71.00, в N2 = 5.45, остаток на след. месяц = 66.02


# **Интерпретация**

* Максимальный суммарный выпуск продукта Y за три месяца составляет 58.2261 единицы.

* Оптимальная стратегия распределения средств:

1. 1 месяц: в цех N1 вложить 86,00 единиц, в цех N2 - 16,00 единиц. Остаток на начало следующего месяца - 88,52.

2. 2 месяц: в цех N1 вложить 82,00 единиц, в цех N2 - 6,52 единиц. Остаток на начало следующего месяца - 76,45.

3. 3 месяц: в цех N1 вложить 71,00 единиц, в цех N2 - 5,45 единиц. Остаток на конец квартала - 66,02 (не используется).

* Полученное распределение учитывает как производительность каждого цеха, так и разную «сохраняемость» средств (коэффициенты остатка 0,86 для N1 и 0,91 для N2), что позволяет максимизировать суммарный выпуск за весь период.